# 기상청_지상(종관, ASOS) 시간자료 조회서비스

https://www.data.go.kr/data/15057210/openapi.do

In [1]:
import requests
import pandas as pd
import time
import os
import glob
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def fetch_resilient_weather_data():
    # 1. 초기 설정
    url = 'http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList'
    # Decoding 인증키 사용 (requests가 자동 인코딩하므로 == 형태가 안전)
    # service_key = 'gR9efoM90FwF0PklBCvwsoDUOCQy8FMzFbsJLI8ARdJHTCPCD32vV40mNCVXUDtO0CjcDfd8rgHQZcMSxhOcmg=='
    service_key = '6214aba2c482f34828e1168919e1b73c533d5472baaf5df3bbf0c140945655ba'
    
    start_year = 1974
    end_year = 1979
    temp_dir = 'asos_raw_data'
    if not os.path.exists(temp_dir): os.makedirs(temp_dir)

    stn_ids = [
        90, 95, 98, 99, 100, 101, 102, 104, 105, 106, 108, 112, 114, 115, 119, 121, 
        127, 129, 130, 131, 133, 135, 136, 137, 138, 140, 143, 146, 152, 155, 156, 
        159, 162, 165, 168, 169, 170, 172, 174, 184, 185, 188, 189, 192, 201, 202, 
        203, 211, 212, 216, 217, 221, 226, 232, 235, 236, 238, 243, 244, 245, 247, 
        248, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 262, 263, 264, 266, 
        271, 272, 273, 276, 277, 278, 279, 281, 283, 284, 285, 288, 289, 294, 295
    ]

    # 세션 및 재시도 설정 (타임아웃 방지 핵심)
    session = requests.Session()
    retry = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    session.mount('http://', HTTPAdapter(max_retries=retry))

    for stn_id in stn_ids:
        stn_file_path = os.path.join(temp_dir, f"STN_{stn_id}.csv")
        if os.path.exists(stn_file_path):
            continue

        print(f"\n[지점 {stn_id}] 수집 시작...")
        stn_frames = []
        
        for year in range(start_year, end_year + 1):
            page_no = 1
            while True:
                params = {
                    'serviceKey': service_key, 'pageNo': str(page_no), 'numOfRows': '700', # 999보다 부하가 적은 700으로 조정
                    'dataType': 'JSON', 'dataCd': 'ASOS', 'dateCd': 'HR',
                    'startDt': f'{year}0101', 'startHh': '00', 'endDt': f'{year}1231', 'endHh': '23',
                    'stnIds': str(stn_id)
                }

                try:
                    # timeout을 60초로 늘리고 세션 사용
                    response = session.get(url, params=params, timeout=60)
                    res_json = response.json()
                    header = res_json.get('response', {}).get('header', {})

                    if header.get('resultCode') == '00':
                        items = res_json.get('response', {}).get('body', {}).get('items', {}).get('item', [])
                        if items:
                            stn_frames.append(pd.DataFrame(items))
                            if len(items) < 700: break
                            page_no += 1
                            time.sleep(0.2) # 속도를 조금 늦춰 서버 차단 방지
                        else: break
                    else:
                        print(f"  - {year}년 오류: {header.get('resultMsg')}")
                        break
                except requests.exceptions.Timeout:
                    print(f"  - [타임아웃] {year}년 {page_no}P 재시도 중...")
                    time.sleep(5) # 타임아웃 시 잠시 대기
                    continue 
                except Exception as e:
                    print(f"  - 예외: {e}")
                    break
        
        if stn_frames:
            pd.concat(stn_frames, ignore_index=True).to_csv(stn_file_path, index=False, encoding='utf-8-sig')
            print(f"  - 지점 {stn_id} 저장 완료")

    # 통합 로직 (기존과 동일)
    print("\n최종 파일 통합 중...")
    all_files = glob.glob(os.path.join(temp_dir, "STN_*.csv"))
    if all_files:
        pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True).to_csv(f"ASOS_HOUR_TOTAL_{start_year}_{end_year}.csv", index=False, encoding='utf-8-sig')
        print("모든 작업 완료!")

if __name__ == "__main__":
    fetch_resilient_weather_data()


[지점 95] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
  - 1976년 오류: NO_DATA
  - 1977년 오류: NO_DATA
  - 1978년 오류: NO_DATA
  - 1979년 오류: NO_DATA

[지점 98] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
  - 1976년 오류: NO_DATA
  - 1977년 오류: NO_DATA
  - 1978년 오류: NO_DATA
  - 1979년 오류: NO_DATA

[지점 99] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
  - 1976년 오류: NO_DATA
  - 1977년 오류: NO_DATA
  - 1978년 오류: NO_DATA
  - 1979년 오류: NO_DATA

[지점 102] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
  - 1976년 오류: NO_DATA
  - 1977년 오류: NO_DATA
  - 1978년 오류: NO_DATA
  - 1979년 오류: NO_DATA

[지점 104] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
  - 1976년 오류: NO_DATA
  - 1977년 오류: NO_DATA
  - 1978년 오류: NO_DATA
  - 1979년 오류: NO_DATA

[지점 106] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
  - 1976년 오류: NO_DATA
  - 1977년 오류: NO_DATA
  - 1978년 오류: NO_DATA
  - 1979년 오류: NO_DATA

[지점 121] 수집 시작...
  - 1974년 오류: NO_DATA
  - 1975년 오류: NO_DATA
  - 1976년 오류: NO_DATA
  - 1977년 오류

In [2]:
import requests
import pandas as pd
import time
import os
import glob
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_station_data(stn_id, start_year, end_year, temp_dir, service_key):
    """
    개별 지점(stn_id)의 전 연도 데이터를 수집하는 단일 스레드 작업 함수 (실시간 로그 포함)
    """
    stn_file_path = os.path.join(temp_dir, f"STN_{stn_id}.csv")
    if os.path.exists(stn_file_path):
        print(f"[스킵] 지점 {stn_id} : 이미 파일이 존재합니다.", flush=True)
        return f"[지점 {stn_id}] 기존 파일이 존재하여 건너뜁니다."

    url = 'http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList'
    stn_frames = []
    
    session = requests.Session()
    retry = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    session.mount('http://', HTTPAdapter(max_retries=retry))

    for year in range(start_year, end_year + 1):
        page_no = 1
        while True:
            params = {
                'serviceKey': service_key, 'pageNo': str(page_no), 'numOfRows': '700',
                'dataType': 'JSON', 'dataCd': 'ASOS', 'dateCd': 'HR',
                'startDt': f'{year}0101', 'startHh': '00', 'endDt': f'{year}1231', 'endHh': '23',
                'stnIds': str(stn_id)
            }

            # [실시간 로그] API 요청 직전에 현재 수집 중인 지점, 연도, 페이지를 출력
            print(f"[수집 중] 지점: {stn_id:3d} | 연도: {year}년 | 페이지: {page_no:2d}P", flush=True)

            try:
                response = session.get(url, params=params, timeout=60)
                res_json = response.json()
                header = res_json.get('response', {}).get('header', {})

                if header.get('resultCode') == '00':
                    items = res_json.get('response', {}).get('body', {}).get('items', {}).get('item', [])
                    if items:
                        stn_frames.append(pd.DataFrame(items))
                        if len(items) < 700: 
                            break
                        page_no += 1
                        time.sleep(0.2)
                    else: 
                        break
                else:
                    print(f"  - [오류] 지점 {stn_id} | {year}년 : {header.get('resultMsg')}", flush=True)
                    break
            except requests.exceptions.Timeout:
                print(f"  - [타임아웃 재시도] 지점 {stn_id} | {year}년 | {page_no}P", flush=True)
                time.sleep(5)
                continue 
            except Exception as e:
                print(f"  - [예외 발생] 지점 {stn_id} | {year}년 : {e}", flush=True)
                break
        
    if stn_frames:
        pd.concat(stn_frames, ignore_index=True).to_csv(stn_file_path, index=False, encoding='utf-8-sig')
        return f"[완료] 지점 {stn_id} : 데이터 저장 완료"
    return f"[실패] 지점 {stn_id} : 수집된 데이터 없음"


def fetch_resilient_weather_data_parallel(max_workers=5):
    """
    ThreadPoolExecutor를 이용해 지점별 병렬 수집을 제어하는 메인 함수
    """
    start_year = 1980
    end_year = 1989
    temp_dir = 'asos_raw_data'
    # service_key = '6214aba2c482f34828e1168919e1b73c533d5472baaf5df3bbf0c140945655ba'
    service_key = 'gR9efoM90FwF0PklBCvwsoDUOCQy8FMzFbsJLI8ARdJHTCPCD32vV40mNCVXUDtO0CjcDfd8rgHQZcMSxhOcmg=='
    
    if not os.path.exists(temp_dir): 
        os.makedirs(temp_dir)

    stn_ids = [
        90, 95, 98, 99, 100, 101, 102, 104, 105, 106, 108, 112, 114, 115, 119, 121, 
        127, 129, 130, 131, 133, 135, 136, 137, 138, 140, 143, 146, 152, 155, 156, 
        159, 162, 165, 168, 169, 170, 172, 174, 184, 185, 188, 189, 192, 201, 202, 
        203, 211, 212, 216, 217, 221, 226, 232, 235, 236, 238, 243, 244, 245, 247, 
        248, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 262, 263, 264, 266, 
        271, 272, 273, 276, 277, 278, 279, 281, 283, 284, 285, 288, 289, 294, 295
    ]

    print(f"병렬 수집 프로세스 시작 (동시 작업 스레드 수: {max_workers})...\n", flush=True)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_stn = {
            executor.submit(fetch_station_data, stn_id, start_year, end_year, temp_dir, service_key): stn_id 
            for stn_id in stn_ids
        }
        
        for future in as_completed(future_to_stn):
            stn_id = future_to_stn[future]
            try:
                result = future.result()
                print(f">>> {result}", flush=True)
            except Exception as e:
                print(f">>> [치명적 에러] 지점 {stn_id} : {e}", flush=True)

    print("\n최종 파일 통합 중...", flush=True)
    all_files = glob.glob(os.path.join(temp_dir, "STN_*.csv"))
    if all_files:
        pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True).to_csv(
            f"ASOS_HOUR_TOTAL_{start_year}_{end_year}.csv", index=False, encoding='utf-8-sig'
        )
        print("모든 작업 완료!", flush=True)
    else:
        print("통합할 CSV 파일이 존재하지 않습니다.", flush=True)

if __name__ == "__main__":
    fetch_resilient_weather_data_parallel(max_workers=5)

병렬 수집 프로세스 시작 (동시 작업 스레드 수: 5)...

[수집 중] 지점:  90 | 연도: 1980년 | 페이지:  1P
[수집 중] 지점:  95 | 연도: 1980년 | 페이지:  1P
[수집 중] 지점:  98 | 연도: 1980년 | 페이지:  1P
[수집 중] 지점:  99 | 연도: 1980년 | 페이지:  1P
[수집 중] 지점: 100 | 연도: 1980년 | 페이지:  1P
  - [오류] 지점 99 | 1980년 : NO_DATA
[수집 중] 지점:  99 | 연도: 1981년 | 페이지:  1P
  - [오류] 지점 98 | 1980년 : NO_DATA  - [오류] 지점 95 | 1980년 : NO_DATA

[수집 중] 지점:  95 | 연도: 1981년 | 페이지:  1P
[수집 중] 지점:  98 | 연도: 1981년 | 페이지:  1P
[수집 중] 지점:  90 | 연도: 1980년 | 페이지:  2P
  - [오류] 지점 98 | 1981년 : NO_DATA
[수집 중] 지점:  98 | 연도: 1982년 | 페이지:  1P
  - [오류] 지점 99 | 1981년 : NO_DATA
[수집 중] 지점:  99 | 연도: 1982년 | 페이지:  1P
[수집 중] 지점: 100 | 연도: 1980년 | 페이지:  2P
  - [오류] 지점 95 | 1981년 : NO_DATA
[수집 중] 지점:  95 | 연도: 1982년 | 페이지:  1P
[수집 중] 지점:  90 | 연도: 1980년 | 페이지:  3P
  - [오류] 지점 95 | 1982년 : NO_DATA
[수집 중] 지점:  95 | 연도: 1983년 | 페이지:  1P
  - [오류] 지점 98 | 1982년 : NO_DATA
[수집 중] 지점:  98 | 연도: 1983년 | 페이지:  1P
[수집 중] 지점: 100 | 연도: 1980년 | 페이지:  3P
[수집 중] 지점:  90 | 연도: 1980년 | 페이지:  4P
  - [오류] 지점 99 | 